<a href="https://colab.research.google.com/github/kagamirudo/CS-614_Application_ML_AI/blob/main/Homework%206/HW6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applications of Machine Learning
## HW 6 - Using LLMs

## Introduction
This assignment will be a bit different that many of our other ones.

We'll continue using existing models, but instead of getting them directly from PyTorch, we'll get them from  *Hugging Face* (https://huggingface.co/)

The first thing you'll need to do is to install transformers from hugging face:
*pip install transformers*

In [13]:
!nvidia-smi

Thu Feb 26 17:43:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             65W /  400W |    1624MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### GPT2
Let's start by getting a *causal LM* model pre-trained using the gpt2 weights as well as a tokenizer, also pretrained using gpt2.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

#### Tokenize Input
Now let's tokenize some input.

In [2]:
input_text = "Today I went to the park and "
input_ids = tokenizer([input_text], return_tensors='pt')  #return as pytorch tensors
print(input_ids)

{'input_ids': tensor([[8888,  314, 1816,  284,  262, 3952,  290,  220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


#### Generate Output
Now let's use the model to generate output

In [3]:
output = model.generate(input_ids['input_ids'], max_length=100)
print(output)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([[8888,  314, 1816,  284,  262, 3952,  290,  220, 1849,   40, 2497,  257,
         1256,  286,  661,  612,   13,  314,  373, 1107, 6568,  284,  766,  644,
          484,  547, 1804,   13,  314,  373, 1107, 6568,  284,  766,  644,  484,
          547, 1804,   13,  314,  373, 1107, 6568,  284,  766,  644,  484,  547,
         1804,   13,  314,  373, 1107, 6568,  284,  766,  644,  484,  547, 1804,
           13,  314,  373, 1107, 6568,  284,  766,  644,  484,  547, 1804,   13,
          314,  373, 1107, 6568,  284,  766,  644,  484,  547, 1804,   13,  314,
          373, 1107, 6568,  284,  766,  644,  484,  547, 1804,   13,  314,  373,
         1107, 6568,  284,  766]])
Today I went to the park and  I saw a lot of people there. I was really excited to see what they were doing. I was really excited to see what they were doing. I was really excited to see what they were doing. I was really excited to see what they were doing. I was really excited to see what they were doing. I was re

### Pipelines
Next we'll look at Hugging Face's *pipelines*

Pipelines provide a simple API to complex models.

https://huggingface.co/docs/transformers/v4.36.1/main_classes/pipelines

Hugging face has pipelines for many different tasks, which we can specify as strings.

Here's some examples:
- "text-classification"
- "text-generation"
- "question-answering"


In [4]:
from transformers import pipeline

#### Our Input
Throughout our examples, we'll utilize the following input text:

In [5]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""
print(text)

Dear Amazon, last week I ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead! As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered. Enclosed are copies of my records concerning this purchase. I expect to hear from you soon. Sincerely, Bumblebee.


#### Text Classification
Let's start with text classification

In [6]:
classifier = pipeline("text-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Let's classify this (as positive or negative)!  

Note, we don't need to explicity tokenize/encode/decode anything.  Everything is *abstracted* by the API

In [7]:
outputs = classifier(text)
print(outputs)

[{'label': 'NEGATIVE', 'score': 0.9015459418296814}]


#### Text Generation
Next let's do some text generation

In [8]:
generator = pipeline("text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We'll input the text from before, adding to it the start of a *customer service response*

In [9]:
response = "Dear Bumblebee, We at Amazon are not sorry at all, "
prompt = text + "\n\nCustomer Service Response:" + response
print(prompt)

Dear Amazon, last week I ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead! As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered. Enclosed are copies of my records concerning this purchase. I expect to hear from you soon. Sincerely, Bumblebee.

Customer Service Response:Dear Bumblebee, We at Amazon are not sorry at all, 


Let's generate the response!

In [10]:
outputs = generator(prompt, max_length=200)
print(outputs)
print(outputs[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Dear Amazon, last week I ordered an Optimus Prime action figure from your online store in Germany. Unfortunately, when I opened the package, I discovered to my horror that I had been sent an action figure of Megatron instead! As a lifelong enemy of the Decepticons, I hope you can understand my dilemma. To resolve the issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered. Enclosed are copies of my records concerning this purchase. I expect to hear from you soon. Sincerely, Bumblebee.\n\nCustomer Service Response:Dear Bumblebee, We at Amazon are not sorry at all, !! We received your orders, but we are not happy that you did not reply to our phone calls. Bumblebee has nothing to do with this. We have an excellent customer service representative, who is also in charge of handling these issues. In the interest of getting the business handled properly, we have been unable to find a way to exchange the Optimus Prime figure. We have contacted your 

#### Question-Answering
Let's wrap up by using the *question-answering* pipeline.

In [11]:
reader = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

To this we can provide the question and the *context*

Note that internally the context is usually *prepended* to the question as input to the LLM

In [12]:
question = "What does the customer want?"
outputs = reader(question=question, context=text)
print(outputs)

{'score': 0.6312918066978455, 'start': 335, 'end': 358, 'answer': 'an exchange of Megatron'}


## Required: Two New Example Runs

The cells below add **two new example runs** for each core idea (GPT-2, text classification, text generation, question answering).

After running them in Colab, **save the notebook** so the inputs/outputs appear in your submitted technical document (and GitHub can render it).

### GPT-2 Examples (two new prompts)

Run this cell to generate **two new GPT-2 completions** from two different prompts.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import torch

# Ensure model/tokenizer exist (in case this section is run standalone)
if "model" not in globals() or "tokenizer" not in globals():
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


def gpt2_complete(prompt: str, *, seed: int, max_new_tokens: int = 60) -> str:
    set_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


prompt_a = "In a future where homework is graded by robots, the student learned that"
prompt_b = "Recipe: To make the perfect grilled cheese, start by"

print("GPT-2 Example 1 prompt:")
print(prompt_a)
print("\nGPT-2 Example 1 output:")
print(gpt2_complete(prompt_a, seed=614, max_new_tokens=60))

print("\n" + "-" * 80 + "\n")

print("GPT-2 Example 2 prompt:")
print(prompt_b)
print("\nGPT-2 Example 2 output:")
print(gpt2_complete(prompt_b, seed=615, max_new_tokens=60))

### Text Classification Examples (two new inputs)

Run this cell to classify **two new pieces of text** as positive/negative.

In [ ]:
from transformers import pipeline

if "classifier" not in globals():
    classifier = pipeline("text-classification")

examples = [
    (
        "Text Classification Example 1 (positive)",
        "I expected this to be mediocre, but it was amazing: fast shipping, great packaging, and it worked perfectly.",
    ),
    (
        "Text Classification Example 2 (negative)",
        "This was a frustrating experience. The product arrived damaged, support never responded, and the replacement also failed.",
    ),
]

for title, txt in examples:
    print(title)
    print("Input:", txt)
    print("Output:", classifier(txt))
    print()

### Text Generation Examples (two new prompts)

Run this cell to generate continuations for **two new prompts** using the text-generation pipeline.

In [ ]:
from transformers import pipeline, set_seed

if "generator" not in globals():
    generator = pipeline("text-generation")

set_seed(616)

prompts = [
    (
        "Text Generation Example 1",
        "Email to a professor:\n\nDear Professor, I wanted to follow up on the midterm review session because",
    ),
    (
        "Text Generation Example 2",
        "Customer Support Chat:\n\nUser: My laptop won't turn on after the update.\nAgent:",
    ),
]

pad_id = getattr(getattr(generator, "tokenizer", None), "eos_token_id", None)

for title, prompt in prompts:
    print(title)
    print("Prompt:\n", prompt)
    out = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        top_p=0.95,
        temperature=0.9,
        pad_token_id=pad_id,
    )
    print("\nOutput:\n", out[0]["generated_text"])
    print("\n" + "-" * 80 + "\n")

### Question-Answering Examples (two new question + context pairs)

Run this cell to answer **two new questions**, each with its own context passage.

In [ ]:
from transformers import pipeline

if "reader" not in globals():
    reader = pipeline("question-answering")

qa_examples = [
    {
        "title": "Question-Answering Example 1",
        "context": (
            "The CS614 course covers classical machine learning as well as modern neural network methods. "
            "Homework 6 focuses on using Hugging Face Transformers pipelines for tasks like classification, generation, and QA. "
            "Students submit a technical document showing new example inputs and outputs."
        ),
        "question": "What does Homework 6 focus on?",
    },
    {
        "title": "Question-Answering Example 2",
        "context": (
            "Ada Lovelace wrote extensive notes on the Analytical Engine, including what is often considered the first algorithm intended for a machine. "
            "Her collaboration with Charles Babbage helped popularize early ideas about general-purpose computing."
        ),
        "question": "Who did Ada Lovelace collaborate with?",
    },
]

for ex in qa_examples:
    print(ex["title"])
    print("Context:", ex["context"]) 
    print("Question:", ex["question"])
    print("Output:", reader(question=ex["question"], context=ex["context"]))
    print()